# 09 · Train a replenishment planner and place one order

This notebook builds a **small** transition log, fits the latent planner, and
shows one `decide`: the cover per SKU, the priority ration, and the order
that ration produced. Smoke settings stay in the cells below (a base of one
episode per arm, with over-order weighted higher, a handful of ticks, a few
fit epochs, a search budget of four).

The published log, the fit, the 150-trial `OrderUpToPolicy` studies
at each transfer-sweep holding rate, and the holdout eval are the commands in the root README section
[Latent replenishment](../README.md#latent-replenishment). This notebook
keeps the smoke log in memory. The fit writes its loss curve under
`runs/jepa-replenishment/fit/`, which is gitignored.


In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

from src.jepa.log import collect_transition_log
from src.jepa.plan import DEFAULT_HORIZON_MULTIPLE, fit_latent
from src.sim.distributions import Constant
from src.sim.runner import Runner
from src.tuning.config import TuningConfig
from src.tuning.episode import make_synthetic_catalog, sample_episode

# Smoke settings. Published counts are the README commands.
SMOKE_LAG = 2
SMOKE_EPISODES = 1
SMOKE_TICKS = DEFAULT_HORIZON_MULTIPLE * SMOKE_LAG + 3
SMOKE_SKUS = 2
SMOKE_EPOCHS = 2
SMOKE_BUDGET = 4


## 1 · Collect a small log

The log mixes all four textbook families (hyperparameters sampled from the
tuning search spaces), the exploratory cover-and-priority policy, never-order,
and over-order. Planning depth varies by episode. Factories stay on
`StaticFactoryPolicy`. The published collector uses a base count of 64
episodes per arm from seed offset `14_000_000`, with over-order weighted
higher, at one holding rate. It samples the training partition of the
catalog. The evaluation partition is held out.


In [ ]:
frame = collect_transition_log(
    n_episodes_each=SMOKE_EPISODES,
    episode_length=SMOKE_TICKS,
    k_active=SMOKE_SKUS,
    delivery_lag=SMOKE_LAG,
)
print("log_rows", len(frame))
print("behaviors", sorted(frame["behavior"].unique()))


## 2 · Fit

`fit_latent` trains the per-SKU predictor on that log and wraps it as a
planner. An epoch is one shuffled pass over the training episodes. The
open-loop unroll is `model_unroll_steps`, defaulting to the planning
horizon rather than the delivery lead time. The encoder's observation is
on-hand, an arrival-age profile, a sales window of its own length, cash,
capacity, and the nominal delivery lead time. The Market multiplier, the
disruption identity, and the calendar are not features. The published fit
uses the same function at lead time 3, embedding width 256 and depth 4,
learning rate 1e-4, the default epoch count, and a search budget of 64.
This notebook does not pass a width, so it stays on the reduced model.
The published command writes checkpoints under
`runs/jepa-replenishment/checkpoints/` and the loss curve under
`runs/jepa-replenishment/fit/`.


In [ ]:
planner = fit_latent(
    frame,
    delivery_lag=SMOKE_LAG,
    epochs=SMOKE_EPOCHS,
    seed=0,
    policy_seed=3,
    budget=SMOKE_BUDGET,
)
print("fitted", type(planner).__name__)


## 3 · One decide becomes an order

Attach the planner to the tuner graph's single intermediate and run a short
episode. Covers are chosen by a factored cross-entropy search (`search`
is `cem`). The priority order is the marginal imagined profit per unit of
shared capacity. The first tick's winning proposal is `chosen`: one cover
per SKU and that priority. `action["order"]` is the quantity the ration
granted. The printed `feature_dim` is the encoder width, independent of
the sampled lead time.


In [ ]:
from src.jepa.dynamics import SALES_WINDOW, feature_dim

config = TuningConfig(
    episode_length=3,
    K_active=SMOKE_SKUS,
    delivery_lag=SMOKE_LAG,
    capacity_dist=Constant(50_000),
    balance_dist=Constant(1_000_000),
    init_stock_pct_dist=Constant(0.0),
    holding_rate=0.01,
    order_fee=50.0,
)
spec = sample_episode(make_synthetic_catalog(8), config, episode_seed=14_000_100)
Runner(spec.scenario, policy_overrides={"S": planner}).run()
step = planner.proposals[0]
print("covers", step["chosen"]["covers"])
print("ration", step["chosen"]["priority"])
print("order", step["action"]["order"])
print("search", planner.search)
print("sales_window", SALES_WINDOW)
print("feature_dim", feature_dim())
print("model_unroll_steps", planner.dynamics.model_unroll_steps)
